In [3]:
%load_ext autoreload
%autoreload 2


from libthesis import pdf_writer, update_layout

In [4]:
import pathlib
import pandas

# Step 1: Load list of all FASTA files originally processed
ensembl_dirs = {
    "Chimpanzee": pathlib.Path("/home/richard/source/madb_data/thesis_data/human_chimp/ensembl_alignments"),
    "Gorilla": pathlib.Path("/home/richard/source/madb_data/thesis_data/human_gorilla/ensembl_alignments"),
    "Macaque": pathlib.Path("/home/richard/source/madb_data/thesis_data/human_macaque/ensembl_alignments"),
}

# Extract original unique_ids from filenames
original_ids = {
    species: {f.stem for f in path.glob("*.fa")}
    for species, path in ensembl_dirs.items()
}

# Step 2: Load actual MADB results
df = pandas.read_csv("../thesis_0.1/df/all_alignment_data.csv")
observed_ids = {
    species: set(df[df["species"] == species]["unique_id"].unique())
    for species in original_ids
}

# Step 3: Create table of matched vs missing
rows = []
for species in original_ids:
    total = len(original_ids[species])
    matched = len(original_ids[species] & observed_ids[species])
    missing = total - matched
    rows.append({
        "Species": species,
        "Expected (from FASTA)": total,
        "Loaded into MADB": matched,
        "Missing": missing
    })

summary_df = pandas.DataFrame(rows)
summary_df.sort_values("Species", inplace=True)

# Strip the .fa suffix from MADB unique IDs
df["normalized_uid"] = df["unique_id"].str.replace(".fa", "", regex=False)

# Rebuild sets
observed_ids = {
    species: set(df[df["species"] == species]["normalized_uid"].unique())
    for species in original_ids
}

# Summary table
summary_rows = []
for species in ["Chimpanzee", "Gorilla", "Macaque"]:
    expected = len(original_ids[species])
    loaded = len(original_ids[species] & observed_ids[species])
    missing = expected - loaded
    summary_rows.append({
        "Species": species,
        "Expected (from FASTA)": expected,
        "Loaded into MADB": loaded,
        "Missing": missing
    })

summary_table = pandas.DataFrame(summary_rows)
print(summary_table.to_markdown(index=False))



| Species    |   Expected (from FASTA) |   Loaded into MADB |   Missing |
|:-----------|------------------------:|-------------------:|----------:|
| Chimpanzee |                     111 |                 34 |        77 |
| Gorilla    |                     111 |                 44 |        67 |
| Macaque    |                     111 |                 62 |        49 |


In [5]:
import pandas as pd
from collections import defaultdict

# Load your data
df = pandas.read_csv("../thesis_0.1/df/all_alignment_data.csv")


# Construct original_ids: replace with the original expected stable_ids per species
original_ids = {
    "Chimpanzee": set(df[df["species"] == "Chimpanzee"]["unique_id"].unique()),
    "Gorilla": set(df[df["species"] == "Gorilla"]["unique_id"].unique()),
    "Macaque": set(df[df["species"] == "Macaque"]["unique_id"].unique()),
}

# Define expected k-mer range
expected_kmers = list(range(10, 100, 5))

# Generate expected combinations
expected_grid = {
    (species, uid, k)
    for species, ids in original_ids.items()
    for uid in ids
    for k in expected_kmers
}

# Create observed combinations from actual data
observed_grid = set(
    tuple(row)
    for row in df[["species", "unique_id", "kmer_size"]].itertuples(index=False, name=None)
)

# Identify missing combinations
missing_grid = expected_grid - observed_grid

# Count missing entries per species and k
missing_counts = defaultdict(int)
for species, uid, k in missing_grid:
    missing_counts[(species, k)] += 1

# Create table from missing_counts
table = pd.DataFrame(0, index=["Chimpanzee", "Gorilla", "Macaque"], columns=expected_kmers)
for (species, k), count in missing_counts.items():
    table.loc[species, k] = count

# Save or view the table
table.to_csv("missing_madb_alignment_counts.csv")
display(table)


,10,15,20,25,30,35,40,45,50,55,60,65,70,75,80,85,90,95
Chimpanzee,29,24,23,23,20,21,21,19,16,15,10,9,7,6,6,6,5,4
Gorilla,38,35,32,27,27,25,24,18,19,20,17,15,10,8,6,4,3,2
Macaque,60,50,49,42,38,31,25,18,13,11,10,11,10,7,7,7,6,7


In [6]:
import pandas as pd
from collections import defaultdict

# Load your alignment CSV
df = pandas.read_csv("../thesis_0.1/df/all_alignment_data.csv")


# Replace this with your original FASTA-derived ID sets
original_ids = {
    "Chimpanzee": set(df[df["species"] == "Chimpanzee"]["unique_id"].unique()),
    "Gorilla": set(df[df["species"] == "Gorilla"]["unique_id"].unique()),
    "Macaque": set(df[df["species"] == "Macaque"]["unique_id"].unique()),
}

# Define the full k-mer range
expected_kmers = list(range(10, 100, 5))

# Create the full grid of expected (species, uid, k)
expected_grid = {
    (species, uid, k)
    for species, uids in original_ids.items()
    for uid in uids
    for k in expected_kmers
}

# Observed (species, uid, k) tuples from the CSV
observed_grid = set(
    tuple(row)
    for row in df[["species", "unique_id", "kmer_size"]].itertuples(index=False, name=None)
)

# Identify missing entries
missing_grid = expected_grid - observed_grid

# Count missing and observed per (species, k)
missing_counts = defaultdict(int)
observed_counts = defaultdict(int)

for species, uid, k in missing_grid:
    missing_counts[(species, k)] += 1

for species, uid, k in observed_grid:
    observed_counts[(species, k)] += 1

# Create DataFrames
species_list = ["Chimpanzee", "Gorilla", "Macaque"]
missing_df = pd.DataFrame(0, index=species_list, columns=expected_kmers)
observed_df = pd.DataFrame(0, index=species_list, columns=expected_kmers)

for (species, k), count in missing_counts.items():
    missing_df.loc[species, k] = count

for (species, k), count in observed_counts.items():
    observed_df.loc[species, k] = count

# Display the results
print("Missing alignments per species and k-mer size:")
display(missing_df)

print("Observed alignments per species and k-mer size:")
display(observed_df)

# Optionally save
missing_df.to_csv("missing_madb_alignment_counts.csv")
observed_df.to_csv("observed_madb_alignment_counts.csv")


Missing alignments per species and k-mer size:


,10,15,20,25,30,35,40,45,50,55,60,65,70,75,80,85,90,95
Chimpanzee,29,24,23,23,20,21,21,19,16,15,10,9,7,6,6,6,5,4
Gorilla,38,35,32,27,27,25,24,18,19,20,17,15,10,8,6,4,3,2
Macaque,60,50,49,42,38,31,25,18,13,11,10,11,10,7,7,7,6,7


Observed alignments per species and k-mer size:


,10,15,20,25,30,35,40,45,50,55,60,65,70,75,80,85,90,95
Chimpanzee,5,10,11,11,14,13,13,15,18,19,24,25,27,28,28,28,29,30
Gorilla,6,9,12,17,17,19,20,26,25,24,27,29,34,36,38,40,41,42
Macaque,2,12,13,20,24,31,37,44,49,51,52,51,52,55,55,55,56,55


In [7]:
# Get shared unique_ids between Chimpanzee and Macaque
chimp_ids = original_ids["Chimpanzee"]
macaque_ids = original_ids["Macaque"]
shared_ids = chimp_ids & macaque_ids

# Build a set of observed (species, uid, k) to filter
observed_set = set(tuple(x) for x in df[["species", "unique_id", "kmer_size"]].itertuples(index=False, name=None))

# Initialize row for shared observed alignments
shared_counts = {k: 0 for k in expected_kmers}

# Count cases where both Chimp and Macaque observed the same unique_id at the same k
for uid in shared_ids:
    for k in expected_kmers:
        chimp_entry = ("Chimpanzee", uid, k)
        macaque_entry = ("Macaque", uid, k)
        if chimp_entry in observed_set and macaque_entry in observed_set:
            shared_counts[k] += 1

# Append to the observed_df
observed_df.loc["Chimp+Macaque"] = pd.Series(shared_counts)

# Show result
print("Observed alignments including shared Chimp+Macaque:")
display(observed_df)

# Optionally export
observed_df.to_csv("observed_with_chimp_macaque_combined.csv")


Observed alignments including shared Chimp+Macaque:


,10,15,20,25,30,35,40,45,50,55,60,65,70,75,80,85,90,95
Chimpanzee,5,10,11,11,14,13,13,15,18,19,24,25,27,28,28,28,29,30
Gorilla,6,9,12,17,17,19,20,26,25,24,27,29,34,36,38,40,41,42
Macaque,2,12,13,20,24,31,37,44,49,51,52,51,52,55,55,55,56,55
Chimp+Macaque,2,10,10,10,13,13,12,13,17,18,22,21,22,22,22,21,22,22


In [8]:
# Generate set of all expected (species, unique_id) pairs from the original FASTAs
expected_pairs = set()
for species, ids in original_ids.items():
    expected_pairs.update((species, uid) for uid in ids)

# Generate a full expected grid: each species × unique_id × k
expected_kmers = list(range(10, 100, 5))
expected_grid = {
    (species, uid, k)
    for species, uid in expected_pairs
    for k in expected_kmers
}

# Actual entries in the CSV
observed_grid = set(
    tuple(row)
    for row in df[["species", "unique_id", "kmer_size"]].itertuples(index=False, name=None)
)

# Missing entries
missing_grid = expected_grid - observed_grid

missing_grid

{('Chimpanzee', 'ENSG00000053371-0.fa', 10),
 ('Chimpanzee', 'ENSG00000053371-0.fa', 15),
 ('Chimpanzee', 'ENSG00000053371-0.fa', 20),
 ('Chimpanzee', 'ENSG00000053371-0.fa', 25),
 ('Chimpanzee', 'ENSG00000053371-0.fa', 30),
 ('Chimpanzee', 'ENSG00000053371-0.fa', 35),
 ('Chimpanzee', 'ENSG00000053371-0.fa', 40),
 ('Chimpanzee', 'ENSG00000053371-0.fa', 45),
 ('Chimpanzee', 'ENSG00000116337.fa', 10),
 ('Chimpanzee', 'ENSG00000116337.fa', 15),
 ('Chimpanzee', 'ENSG00000116337.fa', 20),
 ('Chimpanzee', 'ENSG00000116337.fa', 25),
 ('Chimpanzee', 'ENSG00000116337.fa', 30),
 ('Chimpanzee', 'ENSG00000116337.fa', 35),
 ('Chimpanzee', 'ENSG00000116337.fa', 75),
 ('Chimpanzee', 'ENSG00000116337.fa', 80),
 ('Chimpanzee', 'ENSG00000116337.fa', 85),
 ('Chimpanzee', 'ENSG00000116337.fa', 90),
 ('Chimpanzee', 'ENSG00000116337.fa', 95),
 ('Chimpanzee', 'ENSG00000116863.fa', 10),
 ('Chimpanzee', 'ENSG00000116863.fa', 15),
 ('Chimpanzee', 'ENSG00000116863.fa', 20),
 ('Chimpanzee', 'ENSG00000116863.fa', 

In [9]:
species_colors = {
    "Chimpanzee": "#CC0000",  # Red
    "Gorilla":    "#E69F00",  # Orange-Yellow
    "Macaque":    "#0072B2",  # Blue
}

In [13]:
import plotly.graph_objects as go
import pandas as pd

# Assuming observed_df is already created and has species as index and k-mers as columns
# Also assuming expected_kmers is a sorted list of integers from 10 to 95
expected_kmers = list(range(10, 100, 5))

species_colors = {
    "Chimpanzee": "#CC0000",  # Red
    "Gorilla":    "#E69F00",  # Orange-Yellow
    "Macaque":    "#0072B2",  # Blue
}

# Only include species of interest in plot (you can add "Chimp+Macaque" if desired)
species_to_plot = ["Chimpanzee", "Gorilla", "Macaque"]

fig = go.Figure()

for species in species_to_plot:
    fig.add_trace(go.Bar(
        x=expected_kmers,
        y=observed_df.loc[species, expected_kmers],
        name=species,
        marker_color=species_colors[species]
    ))

fig.update_layout(
    barmode="group",
    xaxis_title="$k\\text{-mer size}$",
    yaxis_title="Successful alignments",
    title="Number of successful MADB alignments by species and k-mer size",
    legend=dict(x=0.01, y=0.99),
    width=800,
    height=400,
    margin=dict(l=40, r=20, t=40, b=40),
    xaxis=dict(
        tickmode="array",
        tickvals=expected_kmers,
        ticktext=[str(k) for k in expected_kmers]
    )
)


# Save
writer = pdf_writer()
writer(fig, "successful_madb_alignments.pdf")
fig.show()

